# Lecture 3 - Business-License-Only Matching Template

This notebook builds a **business-license-only** matching workflow for the sample lease documents.

It does not require parcel geometry and is intended for:
- candidate matching between sample docs and business licenses
- confidence scoring (`high` / `medium` / `low`)
- export of a manual review sheet for QA

## 1 Pipeline Overview

**Goal:** Match 20 sample lease documents (1992–2006) to business licenses by text similarity, without parcel geometry.

**Why block-level prefiltering works:**
- First narrow by zip code & house number (fast index lookup)
- Then apply fuzzy matching on small candidate pool (efficient)
- Result: Much faster than comparing all 1M+ licenses per sample

**Scoring:** final_score = 0.50×address + 0.30×tenant + 0.10×landlord + 0.10×year

**Confidence levels:** high (≥0.70), medium (0.45–0.69), low (<0.45)

In [18]:
from pathlib import Path
import re
import difflib
import pandas as pd

ROOT = Path('.')
SAMPLE_DIR = ROOT / 'Dataset' / 'Samples' / 'sample20_bundle'
SAMPLE_CSV = SAMPLE_DIR / 'lease_answers_sample.csv'
FULL_HEADER_CSV = SAMPLE_DIR / 'lease_answers_full_headers.csv'
LICENSE_CSV = ROOT / 'Dataset' / 'Business_Licenses_20260424.csv'

OUT_DIR = ROOT / 'Outputs' / 'lecture3_business_license_only'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K_PER_SAMPLE = 5
YEAR_WINDOW = 8
MAX_PREFILTER_CANDIDATES = 20000

assert SAMPLE_CSV.exists(), f'Missing sample csv: {SAMPLE_CSV}'
assert LICENSE_CSV.exists(), f'Missing business license csv: {LICENSE_CSV}'

print('Sample CSV:', SAMPLE_CSV)
print('Header CSV:', FULL_HEADER_CSV if FULL_HEADER_CSV.exists() else 'not found (optional)')
print('License CSV:', LICENSE_CSV)
print('Output dir :', OUT_DIR)
print('MAX_PREFILTER_CANDIDATES:', MAX_PREFILTER_CANDIDATES)

Sample CSV: Dataset\Samples\sample20_bundle\lease_answers_sample.csv
Header CSV: Dataset\Samples\sample20_bundle\lease_answers_full_headers.csv
License CSV: Dataset\Business_Licenses_20260424.csv
Output dir : Outputs\lecture3_business_license_only
MAX_PREFILTER_CANDIDATES: 20000


### 1-1 Extracting info from Samples

In [20]:
def norm_text(x: str) -> str:
    if pd.isna(x):
        return ''
    x = str(x).strip().lower()
    x = re.sub(r'\s+', ' ', x)
    return x


def strip_prefix_phrases(x: str) -> str:
    # Remove common extraction prefixes like "The landlord is ..."
    patterns = [
        r'^the\s+landlord\s+is\s+',
        r'^landlord\s*:\s*',
        r'^the\s+tenant\s+is\s+',
        r'^tenant\s*:\s*',
        r'^the\s+address\s+of\s+the\s+property\s+is\s+',
    ]
    out = x
    for p in patterns:
        out = re.sub(p, '', out, flags=re.IGNORECASE)
    return out.strip(' .')


def normalize_address(x: str) -> str:
    x = norm_text(x)
    x = strip_prefix_phrases(x)
    x = re.sub(r'[^a-z0-9\s]', ' ', x)
    x = re.sub(r'\b(illinois|il)\b', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x


def normalize_name(x: str) -> str:
    x = norm_text(x)
    x = strip_prefix_phrases(x)
    x = re.sub(r'[^a-z0-9\s]', ' ', x)
    x = re.sub(r'\b(llc|inc|corp|corporation|co|l l c|ltd|limited|company|trust|association)\b', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x


def parse_doc_year(filepath: str) -> int | None:
    # Example: /.../LEASE/1994/11/94971368.pdf
    m = re.search(r'/(19\d{2}|20\d{2})/', str(filepath))
    return int(m.group(1)) if m else None


def extract_zip5(text: str) -> str:
    if not isinstance(text, str):
        return ''
    m = re.search(r'\b(\d{5})(?:-\d{4})?\b', text)
    return m.group(1) if m else ''


def extract_house_number(addr_norm: str) -> str:
    if not isinstance(addr_norm, str):
        return ''
    m = re.match(r'\s*(\d{1,6})\b', addr_norm)
    return m.group(1) if m else ''


def pick_name_anchor(name_norm: str) -> str:
    if not isinstance(name_norm, str):
        return ''
    tokens = [t for t in name_norm.split() if len(t) >= 4]
    if not tokens:
        return ''
    tokens.sort(key=len, reverse=True)
    return tokens[0]


sample_df = pd.read_csv(SAMPLE_CSV, dtype=str).fillna('')

# Optional readable mapping for Q columns from full header file
q_readable = {}
if FULL_HEADER_CSV.exists():
    full_header = pd.read_csv(FULL_HEADER_CSV, nrows=0).columns.tolist()
    # full header has Q1..Q39 equivalents + filename
    for i in range(1, min(39, len(full_header) - 1) + 1):
        q_readable[f'Q{i}'] = full_header[i - 1]

sample_df['doc_year'] = sample_df['filepath'].map(parse_doc_year)
sample_df['address_raw'] = sample_df.get('Q9', '').astype(str)
sample_df['landlord_raw'] = sample_df.get('Q10', '').astype(str)
sample_df['tenant_raw'] = sample_df.get('Q11', '').astype(str)
sample_df['start_raw'] = sample_df.get('Q12', '').astype(str)
sample_df['end_raw'] = sample_df.get('Q13', '').astype(str)
sample_df['underlying_raw'] = sample_df.get('Q43', '').astype(str)

sample_df['address_norm'] = sample_df['address_raw'].map(normalize_address)
sample_df['landlord_norm'] = sample_df['landlord_raw'].map(normalize_name)
sample_df['tenant_norm'] = sample_df['tenant_raw'].map(normalize_name)
sample_df['zip5'] = sample_df['address_raw'].map(extract_zip5)
sample_df['house_num'] = sample_df['address_norm'].map(extract_house_number)
sample_df['tenant_anchor'] = sample_df['tenant_norm'].map(pick_name_anchor)

# Mark rows with parse errors from upstream extraction
sample_df['has_parse_error'] = sample_df['Q1'].astype(str).str.contains('PARSE_ERROR', case=False, na=False)

print('Sample rows:', len(sample_df))
print('Rows with parse_error:', int(sample_df['has_parse_error'].sum()))
sample_df[['filename', 'doc_year', 'address_raw', 'zip5', 'house_num', 'tenant_anchor', 'has_parse_error']].head(20)

Sample rows: 20
Rows with parse_error: 2


,filename,doc_year,address_raw,zip5,house_num,tenant_anchor,has_parse_error
0,0425239103.pdf,2004,"1746 N. Humboldt Blvd., Chicago, IL, 60647",60647,1746,washer,False
1,94971368.pdf,1994,"190 Lunt, Elk Grove Village, Illinois",,190,,False
2,00308003.pdf,2000,"5130-B West 16th Street, Cicero, Illinois",,5130,industrial,False
3,99393619.pdf,1999,,,,fitness,False
4,95703729.pdf,1995,"1243 Baldwin Lane, Palatine, Illinois",,1243,commercial,False
5,0622050109.pdf,2006,"4228 Oak Park Avenue, Stickney, IL 60402",60402,4228,operating,False
6,0414242205.pdf,2004,"270 East Pearson Street, Chicago, Illinois 60611",60611,270,pearson,False
7,0514441113.pdf,2005,"50 Longmeadow Road, Winnetka, IL 60093",60093,50,,False
8,0325542261.pdf,2003,"840 North Lake Shore Drive, Chicago, Illinois ...",60611,840,shore,False
9,0610802092.pdf,2006,,,,pantry,False


parse_error：
0523103097.pdf
92837966.pdf

### 1-2 Filtering info from Business license csv

#### Load business licenses with selected columns only

In [23]:
# Load business licenses with selected columns only
license_cols = [
    'LICENSE ID', 'LEGAL NAME', 'DOING BUSINESS AS NAME',
    'ADDRESS', 'CITY', 'STATE', 'ZIP CODE',
    'LICENSE DESCRIPTION', 'BUSINESS ACTIVITY',
    'LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE',
    'DATE ISSUED', 'LICENSE STATUS',
    'LATITUDE', 'LONGITUDE'
]

lic_df = pd.read_csv(
    LICENSE_CSV,
    usecols=lambda c: c in license_cols,
    dtype=str,
    low_memory=False
).fillna('')

for c in ['LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE', 'DATE ISSUED']:
    lic_df[c] = pd.to_datetime(lic_df[c], errors='coerce')
for c in ['LATITUDE', 'LONGITUDE']:
    lic_df[c] = pd.to_numeric(lic_df[c], errors='coerce')

lic_df['license_year'] = lic_df['DATE ISSUED'].dt.year
lic_df['zip5'] = lic_df['ZIP CODE'].astype(str).str.extract(r'(\d{5})', expand=False).fillna('')
lic_df['address_norm'] = lic_df['ADDRESS'].map(normalize_address)
lic_df['house_num'] = lic_df['address_norm'].map(extract_house_number)
lic_df['legal_norm'] = lic_df['LEGAL NAME'].map(normalize_name)
lic_df['dba_norm'] = lic_df['DOING BUSINESS AS NAME'].map(normalize_name)
lic_df['desc_norm'] = lic_df['LICENSE DESCRIPTION'].map(norm_text)
lic_df['act_norm'] = lic_df['BUSINESS ACTIVITY'].map(norm_text)

# Build quick block indexes once so we do not scan full table repeatedly
zip_to_idx = lic_df.groupby('zip5').indices
house_to_idx = lic_df.groupby('house_num').indices

print('License rows loaded:', len(lic_df))
print('Unique zip5 blocks :', len(zip_to_idx))
print('Unique house blocks:', len(house_to_idx))
lic_df[['LICENSE ID', 'LEGAL NAME', 'DOING BUSINESS AS NAME', 'ADDRESS', 'zip5', 'house_num', 'LATITUDE', 'LONGITUDE', 'DATE ISSUED']].head(25)

License rows loaded: 1194424
Unique zip5 blocks : 3443
Unique house blocks: 12034


,LICENSE ID,LEGAL NAME,DOING BUSINESS AS NAME,ADDRESS,zip5,house_num,LATITUDE,LONGITUDE,DATE ISSUED
0,3082555,JOE & SONS LAUNDRY LLC,COIN LAUNDRY AND CLEANERS,5885 S ARCHER AVE,60638,5885,41.795637,-87.762129,2026-04-23
1,3082489,"CAL CLOSETS RETAIL, INC.",CALIFORNIA CLOSETS,730 N FRANKLIN ST 110,60654,730,41.895623,-87.635825,2026-04-23
2,3082546,FOREVER HONORING SERVICES LLC,FOREVER HONORING SERVICES LLC,5200 THATCHER RD,60515,5200,NaN,NaN,2026-04-23
3,3076449,GAME NIGHT OUT LLC,GAME NIGHT OUT,2828 N CLARK ST 401,60657,2828,41.933474,-87.645599,2026-04-23
4,3053017,AMANAH ACCOUNTING SERVICE INC,AMANAH ACCOUNTING SERVICE INC,4042 W LAWRENCE AVE,60630,4042,41.968332,-87.729793,2026-04-23
5,3081004,"JAY JAY BHOLE, INC.",SUBWAY,1300 W CERMAK RD 1ST,60608,1300,41.852565,-87.658787,2026-04-23
6,3081512,JONATHON'S SPORTSWEAR INC.,JONATHON'S,11121 S MICHIGAN AVE,60628,11121,41.691937,-87.620895,2026-04-23
7,3071285,"CHICAGO DUFFY, LLC",CHICAGO DUFFY LLC,300 N STATE ST E E,60654,300,41.887558,-87.628153,2026-04-23
8,3077227,2358 CHICAGO INC.,2358 CHICAGO INC.,2358 W CHICAGO AVE 1ST,60622,2358,41.895855,-87.686723,2026-04-23
9,3076701,"Blush Electrolysis & Skincare, Inc",Blush Electrolysis & Skincare,2445 W SUPERIOR ST GROUND,60612,2445,41.894696,-87.688620,2026-04-23


#### !!! Matching Samples ←→Licenses

In [26]:
def ratio(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    return difflib.SequenceMatcher(None, a, b).ratio()


def token_overlap(a: str, b: str) -> float:
    sa = set(a.split()) if a else set()
    sb = set(b.split()) if b else set()
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def year_score(doc_year: int | None, lic_year: float | int | None, window: int = YEAR_WINDOW) -> float:
    if doc_year is None or pd.isna(lic_year):
        return 0.0
    gap = abs(int(doc_year) - int(lic_year))
    if gap > window:
        return 0.0
    return 1.0 - gap / max(window, 1)


def confidence_label(score: float) -> str:
    if score >= 0.70:
        return 'high'
    if score >= 0.45:
        return 'medium'
    return 'low'


def gather_block_candidates(s_row: pd.Series) -> pd.DataFrame:
    idx_sets = []

    z = s_row.get('zip5', '')
    if z and z in zip_to_idx:
        idx_sets.append(set(zip_to_idx[z]))

    h = s_row.get('house_num', '')
    if h and h in house_to_idx:
        idx_sets.append(set(house_to_idx[h]))

    if idx_sets:
        idx_union = set().union(*idx_sets)
        pre = lic_df.loc[list(idx_union)].copy()
    else:
        pre = lic_df.copy()

    # Optional name-anchor refinement to reduce candidate volume
    anchor = s_row.get('tenant_anchor', '')
    if anchor:
        mask_anchor = pre['legal_norm'].str.contains(fr'\b{re.escape(anchor)}\b', regex=True) | pre['dba_norm'].str.contains(fr'\b{re.escape(anchor)}\b', regex=True)
        pre_anchor = pre[mask_anchor]
        if len(pre_anchor) > 0:
            pre = pre_anchor

    # Hard cap to keep runtime predictable; prefer rows with same zip/house if possible
    if len(pre) > MAX_PREFILTER_CANDIDATES:
        pre = pre.head(MAX_PREFILTER_CANDIDATES)

    return pre


rows = []
for _, s in sample_df.iterrows():
    s_addr = s['address_norm']
    s_landlord = s['landlord_norm']
    s_tenant = s['tenant_norm']
    s_year = s['doc_year']

    pre = gather_block_candidates(s)

    if pre.empty:
        rows.append({
            'filename': s['filename'],
            'filepath': s['filepath'],
            'doc_year': s_year,
            'match_rank': 1,
            'confidence': 'low',
            'final_score': 0.0,
            'address_score': 0.0,
            'tenant_score': 0.0,
            'landlord_score': 0.0,
            'year_score': 0.0,
            'LICENSE ID': '',
            'LEGAL NAME': '',
            'DOING BUSINESS AS NAME': '',
            'ADDRESS': '',
            'LICENSE DESCRIPTION': '',
            'BUSINESS ACTIVITY': '',
            'DATE ISSUED': '',
            'LICENSE STATUS': '',
            'candidate_pool_size': 0,
            'has_parse_error': bool(s['has_parse_error']),
            'review_status': 'needs_manual_review',
            'review_notes': 'No license candidate found by block prefilter'
        })
        continue

    score_rows = []
    for i, l in pre.iterrows():
        addr_sc = max(ratio(s_addr, l['address_norm']), token_overlap(s_addr, l['address_norm']))
        tenant_sc = max(ratio(s_tenant, l['legal_norm']), ratio(s_tenant, l['dba_norm']))
        landlord_sc = max(ratio(s_landlord, l['legal_norm']), ratio(s_landlord, l['dba_norm']))
        y_sc = year_score(s_year, l['license_year'])

        final = 0.50 * addr_sc + 0.30 * tenant_sc + 0.10 * landlord_sc + 0.10 * y_sc

        score_rows.append((i, final, addr_sc, tenant_sc, landlord_sc, y_sc))

    score_rows.sort(key=lambda x: x[1], reverse=True)

    for rank, (idx, final, addr_sc, tenant_sc, landlord_sc, y_sc) in enumerate(score_rows[:TOP_K_PER_SAMPLE], start=1):
        l = pre.loc[idx]
        rows.append({
            'filename': s['filename'],
            'filepath': s['filepath'],
            'doc_year': s_year,
            'match_rank': rank,
            'confidence': confidence_label(final),
            'final_score': round(float(final), 4),
            'address_score': round(float(addr_sc), 4),
            'tenant_score': round(float(tenant_sc), 4),
            'landlord_score': round(float(landlord_sc), 4),
            'year_score': round(float(y_sc), 4),
            'LICENSE ID': l['LICENSE ID'],
            'LEGAL NAME': l['LEGAL NAME'],
            'DOING BUSINESS AS NAME': l['DOING BUSINESS AS NAME'],
            'ADDRESS': l['ADDRESS'],
            'LICENSE DESCRIPTION': l['LICENSE DESCRIPTION'],
            'BUSINESS ACTIVITY': l['BUSINESS ACTIVITY'],
            'DATE ISSUED': l['DATE ISSUED'].strftime('%Y-%m-%d') if pd.notna(l['DATE ISSUED']) else '',
            'LICENSE STATUS': l['LICENSE STATUS'],
            'candidate_pool_size': int(len(pre)),
            'has_parse_error': bool(s['has_parse_error']),
            'review_status': 'needs_manual_review',
            'review_notes': ''
        })

match_df = pd.DataFrame(rows)

# Keep one quick summary table for decision making
summary_df = (
    match_df.groupby('filename', as_index=False)
    .agg(best_score=('final_score', 'max'), best_confidence=('confidence', 'first'), candidate_pool_size=('candidate_pool_size', 'max'))
    .sort_values(['best_score', 'filename'], ascending=[False, True])
)

print('Match rows:', len(match_df))
print('Unique sample docs:', match_df['filename'].nunique())
print('Avg candidate pool size:', round(match_df['candidate_pool_size'].mean(), 2) if len(match_df) else 0)
print(summary_df.head(20).to_string(index=False))

Match rows: 94
Unique sample docs: 20
Avg candidate pool size: 3271.95
      filename  best_score best_confidence  candidate_pool_size
0426634094.pdf      0.7949            high                   12
0325542261.pdf      0.6745          medium                   83
  97603790.pdf      0.6405          medium                    2
0414242205.pdf      0.5487          medium                   24
0401204129.pdf      0.5155          medium                 9810
0310515077.pdf      0.5033          medium                 1169
  04068051.pdf      0.4990          medium                  586
0622050109.pdf      0.4671          medium                 1047
  94588820.pdf      0.4510          medium                  562
0610802092.pdf      0.4276             low                 2349
  00308003.pdf      0.4184             low                  196
0425239103.pdf      0.4116             low                    8
  99393619.pdf      0.3898             low                 3196
  97875020.pdf      0.3851       

#### Attach key sample fields for easier manual review

In [31]:
# Attach key sample fields for easier manual review
sample_keep = [
    'filename', 'filepath', 'doc_year', 'has_parse_error',
    'Q9', 'Q10', 'Q11', 'Q12', 'Q13', 'Q15', 'Q40', 'Q41', 'Q42', 'Q43'
]
best_match_df = (
    match_df.sort_values(['filename', 'final_score', 'match_rank'], ascending=[True, False, True])
    .groupby('filename', as_index=False)
    .first()
)
best_match_df = best_match_df.merge(
    sample_df[sample_keep],
    on=['filename', 'filepath', 'doc_year', 'has_parse_error'],
    how='left'
)

# Add license coordinates for best-match export
lic_geo = (
    lic_df[['LICENSE ID', 'LATITUDE', 'LONGITUDE']]
    .drop_duplicates('LICENSE ID')
)
best_match_df = best_match_df.merge(lic_geo, on='LICENSE ID', how='left')

review_df = match_df.merge(sample_df[sample_keep], on=['filename', 'filepath', 'doc_year', 'has_parse_error'], how='left')

match_out = OUT_DIR / 'logs' / 'sample20_business_license_candidates.csv'
summary_out = OUT_DIR / 'logs' / 'sample20_business_license_bestmatch_summary.csv'
best_detail_out = OUT_DIR / 'sample20_best_license_matches.csv'
review_out = OUT_DIR / 'logs' / 'sample20_business_license_review_sheet.csv'

match_df.to_csv(match_out, index=False, encoding='utf-8-sig')
summary_df.to_csv(summary_out, index=False, encoding='utf-8-sig')
best_match_df.to_csv(best_detail_out, index=False, encoding='utf-8-sig')
review_df.to_csv(review_out, index=False, encoding='utf-8-sig')

print('Saved:')
print('-', match_out)
print('-', summary_out)
print('-', best_detail_out)
print('-', review_out)

best_match_df[['filename', 'doc_year', 'final_score', 'confidence', 'LICENSE ID', 'LEGAL NAME', 'DOING BUSINESS AS NAME', 'ADDRESS', 'DATE ISSUED', 'LATITUDE', 'LONGITUDE']].head(20)

Saved:
- Outputs\lecture3_business_license_only\logs\sample20_business_license_candidates.csv
- Outputs\lecture3_business_license_only\logs\sample20_business_license_bestmatch_summary.csv
- Outputs\lecture3_business_license_only\sample20_best_license_matches.csv
- Outputs\lecture3_business_license_only\logs\sample20_business_license_review_sheet.csv


,filename,doc_year,final_score,confidence,LICENSE ID,LEGAL NAME,DOING BUSINESS AS NAME,ADDRESS,DATE ISSUED,LATITUDE,LONGITUDE
0,00308003.pdf,2000,0.4184,low,1215824,TENNIS CORPORATION OF AMERICA,EDENS ATHLETIC CLUB,5130 N CICERO AVE,2002-02-11,41.974461,-87.748120
1,0310515077.pdf,2003,0.5033,medium,1477193,VUONG TUONG,HAPPY NAILS,5041 W FULLERTON AVE,2004-02-19,41.924042,-87.752871
2,0325542261.pdf,2003,0.6745,medium,1366912,"MAHONEY J. HALLBERG, INC.",LAKE SHORE TRAVEL,680 N LAKE SHORE DR 1ST,2003-08-04,41.894798,-87.615867
3,0401204129.pdf,2004,0.5155,medium,1611473,PIT'S RESTAURANT INC.,PIT'S FAST FOOD,6913 S SOUTH CHICAGO AVE,2005-10-25,41.769174,-87.609978
4,04068051.pdf,1994,0.4990,medium,1841526,"ATLAS TRANSLATION SERVICES, INC","ATLAS TRANSLATION SERVICES, INC.",500 W SUPERIOR ST 2802,2007-07-05,41.895609,-87.641579
5,0414242205.pdf,2004,0.5487,medium,1591432,TOWERS VALET SERVICE CENTER,PEARSON VALET,250 E PEARSON ST,2005-04-28,41.897758,-87.619662
6,0425239103.pdf,2004,0.4116,low,1525980,NATIONAL WASHER CORP,NATIONAL WASHER CORP,2215 W WABANSIA AVE 1ST,2004-10-19,41.912531,-87.683154
7,0426634094.pdf,2004,0.7949,high,1505878,MONROE CORP.,MONROE PAVILION HEALTH & TREATMENT CENTER,1400 W MONROE ST 1ST,2004-07-27,41.880357,-87.662082
8,0514441113.pdf,2005,0.3182,low,1568111,DEGABLI CONSTRUCTION CO.,DEGABLI CONSTRUCTION CO.,550 W FRONTAGE RD 3515,2005-11-30,NaN,NaN
9,0523103097.pdf,2005,0.0000,low,3082555,JOE & SONS LAUNDRY LLC,COIN LAUNDRY AND CLEANERS,5885 S ARCHER AVE,2026-04-23,41.795637,-87.762129


## 2 Year-Grouped Preview Maps

Below cell groups all samples by year and creates one interactive map per year.

**Year routing:** Samples from 1992–1999 → use 2000 parcel year; 2000+ → use doc year.

Each marker shows:
- **Tooltip:** License DBA/name | License address | Sample code
- **Popup:** Sample info + License info + Match confidence score

In [ ]:
import folium

PREVIEW_DIR = OUT_DIR / 'sample_preview_maps'
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)


def route_preview_year(doc_year: int | None) -> int | None:
    if doc_year is None or pd.isna(doc_year):
        return None
    doc_year = int(doc_year)
    return 2000 if doc_year < 2000 else doc_year


def parquet_for_preview_year(preview_year: int | None) -> Path | None:
    if preview_year is None:
        return None
    return ROOT / 'Outputs' / 'step2_parcel_land_use' / f'{preview_year}_parcels_land_use_with_license_status.parquet'


def parquet_status(parquet_path: Path | None) -> str:
    if parquet_path is None:
        return 'no_year'
    if not parquet_path.exists():
        return 'missing'
    try:
        _ = pd.read_parquet(parquet_path)
        return 'readable'
    except Exception:
        return 'unreadable'


best_candidates = (
    match_df.sort_values(['filename', 'final_score', 'match_rank'], ascending=[True, False, True])
    .groupby('filename', as_index=False)
    .first()
)

CONF_COLOR = {'high': '#238b45', 'medium': '#f07014', 'low': '#d7301f'}

# ── Step 1: build per-year bucket of (sample_row, best_candidate_row, lic coords) ──
from collections import defaultdict
year_buckets = defaultdict(list)  # preview_year -> list of dicts

for _, row in best_candidates.iterrows():
    sample_row = sample_df.loc[sample_df['filename'] == row['filename']].iloc[0]
    preview_year = route_preview_year(sample_row['doc_year'])

    license_id = str(row.get('LICENSE ID', '')).strip()
    lic_match = (
        lic_df.loc[lic_df['LICENSE ID'].astype(str) == license_id].copy()
        if license_id else pd.DataFrame()
    )
    lic_match = lic_match.loc[lic_match['LATITUDE'].notna() & lic_match['LONGITUDE'].notna()].copy()

    year_buckets[preview_year].append({
        'filename':     row['filename'],
        'doc_year':     sample_row['doc_year'],
        'preview_year': preview_year,
        'final_score':  row.get('final_score', None),
        'confidence':   row.get('confidence', 'low'),
        'LICENSE ID':   row.get('LICENSE ID', ''),
        'ADDRESS':      row.get('ADDRESS', ''),
        'lic_match':    lic_match,
    })

# ── Step 2: one HTML map per preview_year ──
preview_rows = []
for preview_year, items in sorted(year_buckets.items(), key=lambda x: (x[0] is None, x[0])):
    preview_parquet = parquet_for_preview_year(preview_year)
    pq_status       = parquet_status(preview_parquet)

    # Collect all points that have coordinates for this year
    points = [(item, lic_row)
              for item in items
              for _, lic_row in item['lic_match'].head(10).iterrows()]

    year_label = str(preview_year) if preview_year is not None else 'unknown'
    html_path  = PREVIEW_DIR / f'year_{year_label}_preview_map.html'
    html_status = 'skipped_no_point'
    html_note   = ''

    if points:
        all_lats = [float(lr['LATITUDE']) for _, lr in points]
        all_lons = [float(lr['LONGITUDE']) for _, lr in points]
        center_lat = sum(all_lats) / len(all_lats)
        center_lon = sum(all_lons) / len(all_lons)

        m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles='CartoDB positron')

        for item, lic_row in points:
            conf  = str(item.get('confidence', 'low'))
            color = CONF_COLOR.get(conf, '#999999')
            fname = Path(item['filename']).stem   # short sample code name

            legal = lic_row.get('LEGAL NAME', '')
            dba   = lic_row.get('DOING BUSINESS AS NAME', '')
            lic_addr = lic_row.get('ADDRESS', '')

            display_name = dba if dba and dba.strip() else legal

            # Tooltip: license name | license address | sample code
            tooltip_text = f"{display_name} | {lic_addr} | {fname}"

            popup_html = (
                f"<b>Sample:</b> {fname}<br>"
                f"doc_year: {item['doc_year']} &nbsp; preview_year: {preview_year}<br>"
                f"<hr>"
                f"<b>License:</b> {legal}<br>"
                f"DBA: {dba}<br>"
                f"Address: {lic_addr}<br>"
                f"License ID: {lic_row.get('LICENSE ID', '')}<br>"
                f"<hr>"
                f"final_score: {item.get('final_score', '')}<br>"
                f"confidence: <b style='color:{color}'>{conf}</b><br>"
                f"parquet_status: {pq_status}"
            )

            folium.CircleMarker(
                location=[float(lic_row['LATITUDE']), float(lic_row['LONGITUDE'])],
                radius=7,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.75,
                popup=folium.Popup(popup_html, max_width=450),
                tooltip=tooltip_text,
            ).add_to(m)

        folium.LayerControl().add_to(m)
        m.save(html_path)
        html_status = 'done_point_preview'
        html_note = f'Year-grouped map; {len(items)} samples, {len(points)} license points; parquet={pq_status}'

    # Record one row per sample pointing to the shared year map
    for item in items:
        has_point = not item['lic_match'].empty
        preview_rows.append({
            'filename':              item['filename'],
            'doc_year':              item['doc_year'],
            'preview_year':          preview_year,
            'best_score':            item.get('final_score'),
            'best_confidence':       item.get('confidence'),
            'LICENSE ID':            item.get('LICENSE ID', ''),
            'ADDRESS':               item.get('ADDRESS', ''),
            'preview_parquet':       str(preview_parquet) if preview_parquet is not None else '',
            'preview_parquet_status': pq_status,
            'html_preview_path':     str(html_path) if html_status == 'done_point_preview' else '',
            'html_status':           html_status if has_point else 'skipped_no_point',
            'notes':                 html_note or ('' if has_point else 'No license point with coordinates'),
        })

preview_df = (
    pd.DataFrame(preview_rows)
    .sort_values(['preview_year', 'filename'], na_position='last')
    .reset_index(drop=True)
)
preview_csv = OUT_DIR / 'logs'/ 'sample20_preview_map_plan.csv'
preview_df.to_csv(preview_csv, index=False, encoding='utf-8-sig')

print('Saved preview plan:', preview_csv)
print('\nYear-grouped HTML files written:')
for yy in sorted(year_buckets.keys(), key=lambda x: (x is None, x)):
    ypath = PREVIEW_DIR / f'year_{yy}_preview_map.html'
    n_samples = len(year_buckets[yy])
    print(f'  {ypath.name}  ({n_samples} samples)')

print('\nPreview parquet status counts:')
print(preview_df['preview_parquet_status'].value_counts(dropna=False).to_string())
print('\nHTML status counts:')
print(preview_df['html_status'].value_counts(dropna=False).to_string())

preview_df[['filename', 'doc_year', 'preview_year', 'best_score', 'best_confidence',
            'preview_parquet_status', 'html_status', 'html_preview_path']].head(20)

Saved preview plan: Outputs\lecture3_business_license_only\sample20_preview_map_plan.csv

Year-grouped HTML files written:
  year_2000_preview_map.html  (10 samples)
  year_2003_preview_map.html  (2 samples)
  year_2004_preview_map.html  (4 samples)
  year_2005_preview_map.html  (2 samples)
  year_2006_preview_map.html  (2 samples)

Preview parquet status counts:
preview_parquet_status
readable    14
missing      6

HTML status counts:
html_status
done_point_preview    17
skipped_no_point       3


,filename,doc_year,preview_year,best_score,best_confidence,preview_parquet_status,html_status,html_preview_path
0,00308003.pdf,2000,2000,0.4184,low,readable,done_point_preview,Outputs\lecture3_business_license_only\sample_...
1,04068051.pdf,1994,2000,0.4990,medium,readable,done_point_preview,Outputs\lecture3_business_license_only\sample_...
2,92837966.pdf,1992,2000,0.0000,low,readable,done_point_preview,Outputs\lecture3_business_license_only\sample_...
3,94588820.pdf,1994,2000,0.4510,medium,readable,done_point_preview,Outputs\lecture3_business_license_only\sample_...
4,94971368.pdf,1994,2000,0.3364,low,readable,done_point_preview,Outputs\lecture3_business_license_only\sample_...
5,95703729.pdf,1995,2000,0.3555,low,readable,done_point_preview,Outputs\lecture3_business_license_only\sample_...
6,97603790.pdf,1997,2000,0.6405,medium,readable,done_point_preview,Outputs\lecture3_business_license_only\sample_...
7,97875020.pdf,1997,2000,0.3851,low,readable,done_point_preview,Outputs\lecture3_business_license_only\sample_...
8,97962965.pdf,1997,2000,0.1845,low,readable,skipped_no_point,Outputs\lecture3_business_license_only\sample_...
9,99393619.pdf,1999,2000,0.3898,low,readable,done_point_preview,Outputs\lecture3_business_license_only\sample_...


## 3 Combined Global Match Map

All 20 sample best-match license points on one map, color-coded by confidence.

**Color legend:**
- **Green:** High confidence (score ≥ 0.70) — strong match
- **Orange:** Medium confidence (0.45–0.69) — partial match
- **Red:** Low confidence (< 0.45) — weak match
- **Gray pin:** No license coordinates found

Each marker includes license address and year of issue in the hover tooltip.

In [28]:
import folium
from folium.plugins import MarkerCluster

CONFIDENCE_COLOR = {
    'high':   '#238b45',
    'medium': '#f07014',
    'low':    '#d7301f',
}

# Merge best_candidates with sample address/tenant info
sample_extra = sample_df[['filename', 'address_raw', 'tenant_raw']].drop_duplicates('filename')
map_df = best_candidates.merge(sample_extra, on='filename', how='left')
map_df['_preview_year'] = map_df['doc_year'].map(route_preview_year)

# Attach license lat/lon, address, and issue year by LICENSE ID
lic_coords = lic_df[lic_df['LATITUDE'].notna() & lic_df['LONGITUDE'].notna()].copy()
lic_coords['license_year'] = lic_coords['DATE ISSUED'].dt.year
lic_coords = lic_coords[['LICENSE ID', 'LATITUDE', 'LONGITUDE', 'ADDRESS', 'license_year']].drop_duplicates('LICENSE ID')

map_df = map_df.merge(lic_coords, on='LICENSE ID', how='left', suffixes=('_sample', '_lic'))

# Determine map center
has_coords = map_df['LATITUDE'].notna() & map_df['LONGITUDE'].notna()
if has_coords.any():
    center_lat = float(map_df.loc[has_coords, 'LATITUDE'].mean())
    center_lon = float(map_df.loc[has_coords, 'LONGITUDE'].mean())
else:
    center_lat, center_lon = 41.8781, -87.6298  # Chicago fallback

combined_map = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='CartoDB positron')

layer_high      = folium.FeatureGroup(name='High confidence',            show=True)
layer_medium    = folium.FeatureGroup(name='Medium confidence',          show=True)
layer_low       = folium.FeatureGroup(name='Low confidence',             show=True)
layer_no_coords = folium.FeatureGroup(name='No coordinates (skipped)',   show=True)

for _, row in map_df.iterrows():
    conf = str(row.get('confidence', 'low'))
    color = CONFIDENCE_COLOR.get(conf, '#999999')
    lic_year = int(row['license_year']) if pd.notna(row.get('license_year')) else ''
    lic_addr = row.get('ADDRESS_lic', '')

    popup_html = (
        f"<b>{row.get('filename', '')}</b><br>"
        f"doc_year: {row.get('doc_year', '')}<br>"
        f"preview_year: {row.get('_preview_year', '')}<br>"
        f"sample_address: {row.get('address_raw', '')}<br>"
        f"sample_tenant: {row.get('tenant_raw', '')}<br>"
        f"<hr>"
        f"<b>License Match:</b><br>"
        f"license_id: {row.get('LICENSE ID', '')}<br>"
        f"legal_name: {row.get('LEGAL NAME', '')}<br>"
        f"dba_name: {row.get('DOING BUSINESS AS NAME', '')}<br>"
        f"license_address: {lic_addr}<br>"
        f"license_year: {lic_year}<br>"
        f"<hr>"
        f"final_score: {row.get('final_score', '')}<br>"
        f"confidence: <b style='color:{color}'>{conf}</b>"
    )

    if pd.notna(row.get('LATITUDE')) and pd.notna(row.get('LONGITUDE')):
        marker = folium.CircleMarker(
            location=[float(row['LATITUDE']), float(row['LONGITUDE'])],
            radius=8,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.75,
            popup=folium.Popup(popup_html, max_width=450),
            tooltip=f"{row.get('filename', '')} | {lic_addr[:30]} | {lic_year}"
        )
        if conf == 'high':
            layer_high.add_child(marker)
        elif conf == 'medium':
            layer_medium.add_child(marker)
        else:
            layer_low.add_child(marker)
    else:
        folium.Marker(
            location=[center_lat + 0.001, center_lon + 0.001],
            icon=folium.Icon(color='gray', icon='question-sign'),
            popup=folium.Popup(popup_html + '<br><i>No license coordinates available</i>', max_width=450),
            tooltip=f"{row.get('filename', '')} | no coords"
        ).add_to(layer_no_coords)

combined_map.add_child(layer_high)
combined_map.add_child(layer_medium)
combined_map.add_child(layer_low)
combined_map.add_child(layer_no_coords)
folium.LayerControl(collapsed=False).add_to(combined_map)

combined_html = OUT_DIR / 'sample20_all_matches_combined_map.html'
combined_map.save(combined_html)
print('Saved combined map:', combined_html)
print(f'  high:      {(map_df["confidence"] == "high").sum()} samples')
print(f'  medium:    {(map_df["confidence"] == "medium").sum()} samples')
print(f'  low:       {(map_df["confidence"] == "low").sum()} samples')
print(f'  no coords: {(~has_coords).sum()} samples')

combined_map

Saved combined map: Outputs\lecture3_business_license_only\sample20_all_matches_combined_map.html
  high:      1 samples
  medium:    8 samples
  low:       11 samples
  no coords: 3 samples


## 4 Output & Next Steps

- **sample20_best_license_matches.csv** - Best match License + sample 

**logs files** (for manual review):
- `sample20_business_license_candidates.csv` — All top-5 candidates per sample
- `sample20_business_license_bestmatch_summary.csv` — One best match per sample
- `sample20_business_license_review_sheet.csv` — Best match + sample field columns

**Interactive HTML maps:**
- `sample20_all_matches_combined_map.html` — All 20 samples on one map (color by confidence)
- `year_YYYY_preview_map.html` (5 maps) — Sample clusters by year, each marker shows license info

**Next:** Open the HTML files in a browser to explore matches interactively. Use CSV files for QA and record-level validation.